In [3]:
 import torch
 import numpy as np
 import pandas as pd
 import torch.nn as nn
 from torch.utils.data import Dataset, DataLoader

In [4]:
 dataset = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

In [5]:
dataset.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


**Tokanize**

In [6]:
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

**Vocab**

In [7]:
vocab = {'<UNK>': 0}

In [8]:
def build_vocab(row):
  tokanized_question = tokenize(row['question'])
  tokanized_answer = tokenize(row['answer'])

  merged_tokens = tokanized_question + tokanized_answer

  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [9]:
dataset.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [10]:
len(vocab)

324

In [11]:
def text_to_indices(text, vocab):
  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [12]:
text_to_indices('What is your name?', vocab)

[1, 2, 0, 0]

In [13]:
class CustomDataset(Dataset):
  def __init__(self, dataset, vocab):
    self.dataset = dataset
    self.vocab = vocab

  def __len__(self):
    return self.dataset.shape[0]

  def __getitem__(self, index):
    numerical_question = text_to_indices(self.dataset.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.dataset.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [14]:
dataset = CustomDataset(dataset, vocab)

In [15]:
dataset[9]

(tensor([ 1,  2,  3, 50, 51, 19,  3, 45]), tensor([52]))

In [16]:
dataloader = DataLoader(dataset, batch_size= 1, shuffle=True)

In [36]:
for question, answer in dataloader:
  print(answer)

tensor([[280]])
tensor([[112]])
tensor([[238]])
tensor([[9]])
tensor([[273]])
tensor([[276]])
tensor([[207]])
tensor([[205]])
tensor([[100]])
tensor([[52]])
tensor([[160]])
tensor([[188]])
tensor([[121]])
tensor([[259]])
tensor([[68]])
tensor([[128]])
tensor([[179]])
tensor([[321]])
tensor([[132]])
tensor([[102]])
tensor([[77]])
tensor([[205]])
tensor([[145]])
tensor([[287]])
tensor([[41]])
tensor([[131]])
tensor([[36]])
tensor([[268]])
tensor([[61]])
tensor([[185]])
tensor([[32]])
tensor([[317]])
tensor([[49]])
tensor([[16]])
tensor([[149]])
tensor([[36]])
tensor([[65]])
tensor([[249]])
tensor([[113]])
tensor([[7]])
tensor([[194]])
tensor([[191]])
tensor([[154]])
tensor([[215]])
tensor([[298]])
tensor([[85]])
tensor([[136]])
tensor([[6]])
tensor([[23]])
tensor([[199]])
tensor([[225]])
tensor([[53]])
tensor([[254]])
tensor([[98]])
tensor([[311]])
tensor([[307]])
tensor([[106]])
tensor([[121]])
tensor([[72]])
tensor([[95]])
tensor([[184]])
tensor([[85]])
tensor([[155]])
tensor([[134]])


In [30]:
class MyNeuralNetwork(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question  = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))
    return output

In [31]:
epochs = 25
lr = 0.001

In [32]:
model = MyNeuralNetwork(len(vocab))

In [33]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [35]:
for epoch in range(epochs):
  total_loss = 0
  for question, answer in dataloader:
    output = model(question)
    optimizer.zero_grad()
    loss = loss_function(output,answer[0])
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  print(f"Epoch: {epoch+1}, Total loss: {total_loss:4f}")

Epoch: 1, Total loss: 521.485653
Epoch: 2, Total loss: 453.681997
Epoch: 3, Total loss: 375.650700
Epoch: 4, Total loss: 313.468077
Epoch: 5, Total loss: 261.524880
Epoch: 6, Total loss: 213.973900
Epoch: 7, Total loss: 170.722706
Epoch: 8, Total loss: 133.124372
Epoch: 9, Total loss: 102.433858
Epoch: 10, Total loss: 79.299489
Epoch: 11, Total loss: 61.545488
Epoch: 12, Total loss: 48.186433
Epoch: 13, Total loss: 38.459241
Epoch: 14, Total loss: 31.184840
Epoch: 15, Total loss: 25.583787
Epoch: 16, Total loss: 21.480223
Epoch: 17, Total loss: 18.038496
Epoch: 18, Total loss: 15.482844
Epoch: 19, Total loss: 13.085555
Epoch: 20, Total loss: 11.429027
Epoch: 21, Total loss: 9.926864
Epoch: 22, Total loss: 8.727247
Epoch: 23, Total loss: 7.769338
Epoch: 24, Total loss: 6.928485
Epoch: 25, Total loss: 6.230375


In [37]:
def predict(model, question, threshold=0.5):
  numerical_question = text_to_indices(question, vocab)
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)
  output = model(question_tensor)
  probability_output = torch.nn.functional.softmax(output, dim=1)
  value, index = torch.max(probability_output, dim = 1)

  if value < threshold:
    print("I don't know.")
  else:
    print(list(vocab.keys())[index])

In [40]:
predict(model,"What is the capital of France?")

paris


In [41]:
predict(model, "What is my name?")

I don't know.
